In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
#import seaborn as sns
import random
import math
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [ ]:
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/SHMOLLI_VAE_16_output/percentiles_T1_latent_spaces.csv")

In [ ]:
latent_dimensions.rename(columns={'Patient_ID': 'IID'}, inplace=True)
latent_dimensions["IID"] = latent_dimensions["IID"].str[:7]
latent_dimensions["IID"] = latent_dimensions["IID"].astype(int)
latent_dimensions = latent_dimensions.drop_duplicates(subset=['IID'], keep='first')

latent_dimensions = latent_dimensions[['IID',
        'Latent_0', 'Latent_1', 'Latent_2', 'Latent_3', 'Latent_4', 'Latent_5', 'Latent_6', 'Latent_7', 
        'Latent_8', 'Latent_9', 'Latent_10', 'Latent_11', 'Latent_12', 'Latent_13', 'Latent_14', 'Latent_15']]

### T1 Phenotypes GWAS Prep

In [ ]:
covariates = pd.read_csv("./data/ukbb_code/anna_code/preprocessing/covariates.txt", delimiter='\t')

In [ ]:
T1_phenotypes = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_T1_percentiles_HHregressed.csv")

In [ ]:
T1_phenotypes.rename(columns={'id': 'IID'}, inplace=True)
T1_phenotypes['IID'] = T1_phenotypes['IID'].astype(str)

In [ ]:
T1_phenotypes["IID"] = T1_phenotypes["IID"].str[:7]
T1_phenotypes = T1_phenotypes.drop(columns=['Unnamed: 0'])

In [ ]:
def prepare_covarites(phenotypes, covariates, save_name):
    phenotypes_no_duplicates = phenotypes.drop_duplicates(subset=['IID'], keep='first')
    phenotypes_no_duplicates['IID'] = phenotypes_no_duplicates['IID'].astype(int)
    
    # Perform an inner join on 'id' and 'IID' to ensure both DataFrames have the exact same rows
    merged_df = pd.merge(phenotypes_no_duplicates, covariates, on='IID')
    merged_df.replace(-1000, np.nan, inplace=True)
    merged_df = merged_df.dropna()
    
    #Separate the merged DataFrame into trimmed DataFrames
    phenotypes_trimmed = merged_df[['FID', 'IID',
        'Latent_0', 'Latent_1', 'Latent_2', 'Latent_3', 'Latent_4', 'Latent_5', 'Latent_6', 'Latent_7', 
        'Latent_8', 'Latent_9', 'Latent_10', 'Latent_11', 'Latent_12', 'Latent_13', 'Latent_14', 'Latent_15']] # UPDATE THIS
    #phenotypes_trimmed = merged_df[['FID', 'IID', 'Mean_T1', 'T1_Standard_Deviation', 
#                                    'T1_0.25th_Percentile', 'T1_1th_Percentile', 'T1_25th_Percentile', 
#                                    'T1_50th_Percentile', 'T1_75th_Percentile', 'T1_99th_Percentile', 
#                                    'T1_99.75th_Percentile']]

    #phenotypes_trimmed = merged_df[['FID', 'IID', 'Mean_T1', 'T1_Standard_Deviation', 'T1_0.25th_Percentile',
#        'T1_1th_Percentile', 'T1_25th_Percentile', 'T1_50th_Percentile',
#        'T1_75th_Percentile', 'T1_99th_Percentile', 'T1_99.75th_Percentile',
#        'hypertension_status', 'hematocrit', 'Mean_T1_regressed_hematocrit',
#        'T1_Standard_Deviation_regressed_hematocrit',
#        'T1_0.25th_Percentile_regressed_hematocrit',
#        'T1_1th_Percentile_regressed_hematocrit',
#        'T1_25th_Percentile_regressed_hematocrit',
#        'T1_50th_Percentile_regressed_hematocrit',
#        'T1_75th_Percentile_regressed_hematocrit',
#        'T1_99th_Percentile_regressed_hematocrit',
#        'T1_99.75th_Percentile_regressed_hematocrit',
#        'Mean_T1_regressed_hematocrit_hypertension_status',
#        'T1_Standard_Deviation_regressed_hematocrit_hypertension_status',
#        'T1_0.25th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_1th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_25th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_50th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_75th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_99th_Percentile_regressed_hematocrit_hypertension_status',
#        'T1_99.75th_Percentile_regressed_hematocrit_hypertension_status']]
    
    covariates_trimmed = merged_df[list(covariates.columns)]

    # Save phenotypes_trimmed to a CSV file         UPDATE THESE PATHS
    phenotypes_trimmed.to_csv(f'./data/shriya/SHMOLLI-output-unet-myocardium-update2/{save_name}_phenotypes_trimmed.txt', sep='\t', index=False)
    
    # Save covariates_trimmed to a CSV file         UPDATE THESE PATHS
    covariates_trimmed.to_csv(f'./data/shriya/SHMOLLI-output-unet-myocardium-update2/{save_name}_covariates_trimmed.txt', sep='\t', index=False)

In [ ]:
prepare_covarites(latent_dimensions, covariates, "cleaned_latent_dimentions.csv")

In [ ]:
patient_mapping = pd.read_table("./data/shriya/ukb22282_24983_mapping.tsv", header = None)

# Function to get the value from column 0 based on the ID in column 1
def get_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[1].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[1] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return mapping_df.loc[row_index, 0]
    else:
        return None  # Return None if the ID is not found

# Function to get the value from column 0 based on the ID in column 1
def reverse_mapped_ID(id_to_find, mapping_df):
    # Check if the ID exists in column 1
    if id_to_find in mapping_df[0].values:
        # Find the index of the row where the ID is located
        row_index = mapping_df[mapping_df[0] == id_to_find].index[0]
        # Return the value from column 0 in the same row
        return int(mapping_df.loc[row_index, 1])
    else:
        return None  # Return None if the ID is not found

In [ ]:
import pandas as pd

phenotypes = './data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions.no_outliers.residuals.qnorm.txt'
imputed_phenotypes = './data/shriya/SHMOLLI-output-unet-myocardium-update2/cleaned_latent_dimentions_imputed.no_outliers.residuals.qnorm.txt'

imputed_residuals = pd.read_csv(phenotypes, sep='\t') 
print(imputed_residuals)

for index, row in imputed_residuals.iterrows():
    IID = row["IID"]
    mapped_IID = reverse_mapped_ID(int(IID), patient_mapping)

    imputed_residuals.at[index, "IID"] = mapped_IID
    imputed_residuals.at[index, "FID"] = mapped_IID

print(imputed_residuals)

imputed_residuals = imputed_residuals.dropna()

imputed_residuals.to_csv(imputed_phenotypes, sep='\t', index=False)

# Medication and Disease Regression

In [ ]:
medications_data = pd.read_csv("./data/shriya/medications.csv")
labelled_pheno_data = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/cleaned_mean_T1_allpheno.csv", sep=',', header=0)
quality_control = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/mean_T1.csv")


In [ ]:
type1_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E10", na=False)]["id"])
type2_diabetes_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("E11", na=False)]["id"])

MI_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I21", na=False)]["id"])
DCM_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I42", na=False)]["id"])

sarcoidosis_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("D86", na=False)]["id"])
chronic_kidney_disease_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("N18", na=False)]["id"])
hypertension_patients = list(labelled_pheno_data[labelled_pheno_data["icd10"].astype(str).str.contains("I10", na=False)]["id"])

In [ ]:
beta_blocker_patterns = ['acebutolol', 'atenolol', 'betaxolol', 'bisoprolol', 'carvedilol', 
                         'esmolol', 'labetalol', 'metoprolol', 'nadolol', 'nebivolol', 
                         'oxprenolol', 'pindolol', 'propranolol', 'sotalol', 'timolol']

ace_inhibitor_patterns = ['benazepril', 'captopril', 'enalapril', 'fosinopril', 'lisinopril',
                         'moexipril', 'perindopril', 'quinapril', 'ramipril', 'trandolapril']

arb_patterns = ['azilsartan', 'candesartan', 'eprosartan', 'irbesartan', 'losartan',
               'olmesartan', 'telmisartan', 'valsartan']

statin_patterns = ['atorvastatin', 'fluvastatin', 'lovastatin', 'pitavastatin',
                  'pravastatin', 'rosuvastatin', 'simvastatin']

mra_patterns = ['eplerenone', 'finerenone', 'spironolactone']

def get_med_patients(pattern_list, medications_data):
    patient_IDs = []
    for index, row in medications_data.iterrows():        
        # Skip NaN values
        if isinstance(row['Treatment/medication code | Instance 0'], float) and math.isnan(row['Treatment/medication code | Instance 0']):
            continue

        med_entry = str(row['Treatment/medication code | Instance 0']) + "|" + \
           str(row['Treatment/medication code | Instance 1']) + "|" + \
           str(row['Treatment/medication code | Instance 2']) + "|" + \
           str(row['Treatment/medication code | Instance 3'])
        
        # Handle string representation of lists
        med_list = [item for item in med_entry.split('|')]
            
        # Check each medication against patterns
        for med_item in med_list:
            if med_item and any(pattern.lower() in med_item.lower() for pattern in pattern_list):
                patient_IDs.append(row['Participant ID'])
                break

    return patient_IDs

In [ ]:
beta_blocker_patients = get_med_patients(beta_blocker_patterns, medications_data)
ace_patients = get_med_patients(ace_inhibitor_patterns, medications_data)
arb_patients = get_med_patients(arb_patterns, medications_data)
statin_patients = get_med_patients(statin_patterns, medications_data)
mra_patients = get_med_patients(mra_patterns, medications_data)

In [ ]:
# All of the medication and disease state covarites

medications_disease_covar = medications_data.copy()
medications_disease_covar = medications_disease_covar.drop(medications_disease_covar.columns[[1, 2, 3, 4, 5, 6, 7, 8, 9]], axis=1)
medications_disease_covar = medications_disease_covar.rename(columns={"Participant ID" : "IID"})

med_dict = {
    'type1_diabetes': type1_diabetes_patients,
    'type2_diabetes': type2_diabetes_patients,
    'MI': MI_patients,
    'chronic_kidney_disease': chronic_kidney_disease_patients,
    'beta_blocker': beta_blocker_patients,
    'ace_inhibitor': ace_patients,
    'arb': arb_patients,
    'statin': statin_patients,
    'mra': mra_patients
}

# Now loop through the dictionary properly
for med_class, patient_ids in med_dict.items():
    medications_disease_covar[f'{med_class}_true'] = medications_disease_covar['IID'].isin(patient_ids).astype(int)

In [ ]:
medications_disease_covar.to_csv('./data/shriya/SHMOLLI-output-unet-myocardium-update/medications_disease_covaraites.csv', index=False)

### Using the covariate table

In [ ]:
medications_disease_covar = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/medications_disease_covaraites.csv")

In [ ]:
unet_t1_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_all_phenotypes_PheWAS.no_outliers.residuals.qnorm.txt", sep='\t')
latent_dimensions = pd.read_csv("./data/shriya/SHMOLLI_VAE_16_output/latent_dimensions_16_allpatients_PheWAS.no_outliers.residuals.qnorm.txt", sep='\t')
unet_t1_2itr_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_2itr_erroded.no_outliers.residuals.qnorm.txt", sep='\t')
unet_t1_4itr_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_4itr_erroded.no_outliers.residuals.qnorm.txt", sep='\t')
unet_t1_6itr_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_6itr_erroded.no_outliers.residuals.qnorm.txt", sep='\t')
unet_t1_8itr_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_8itr_erroded.no_outliers.residuals.qnorm.txt", sep='\t')
unet_t1_10itr_regressed = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_10itr_erroded.no_outliers.residuals.qnorm.txt", sep='\t')

In [ ]:
quality_control = pd.read_csv("./data/shriya/SHMOLLI-output-unet-myocardium/mean_T1.csv")
bad_images = list(quality_control[quality_control["quality_controlled"] == False]["id"])

unet_t1_regressed = unet_t1_regressed[~unet_t1_regressed["IID"].isin(bad_images)]
latent_dimensions = latent_dimensions[~latent_dimensions["IID"].isin(bad_images)]
unet_t1_2itr_regressed = unet_t1_2itr_regressed[~unet_t1_2itr_regressed["IID"].isin(bad_images)]
unet_t1_4itr_regressed = unet_t1_4itr_regressed[~unet_t1_4itr_regressed["IID"].isin(bad_images)]
unet_t1_6itr_regressed = unet_t1_6itr_regressed[~unet_t1_6itr_regressed["IID"].isin(bad_images)]
unet_t1_8itr_regressed = unet_t1_8itr_regressed[~unet_t1_8itr_regressed["IID"].isin(bad_images)]
unet_t1_10itr_regressed = unet_t1_10itr_regressed[~unet_t1_10itr_regressed["IID"].isin(bad_images)]


In [ ]:
def regress_out_covariates(pheno_df, covar_df, phenotype_cols, covariate_cols, id_col='IID'):
    if id_col not in pheno_df.columns:
        raise ValueError(f"ID column '{id_col}' not found in phenotype dataframe")
    if id_col not in covar_df.columns:
        raise ValueError(f"ID column '{id_col}' not found in covariate dataframe")
    
    # Check if covariates exist in covariate dataframe
    missing_cols = [col for col in covariate_cols if col not in covar_df.columns]
    if missing_cols:
        raise ValueError(f"Covariates not found in covariate dataframe: {missing_cols}")
    
    # Check if phenotypes exist in phenotype dataframe
    missing_phenos = [col for col in phenotype_cols if col not in pheno_df.columns]
    if missing_phenos:
        raise ValueError(f"Phenotypes not found in phenotype dataframe: {missing_phenos}")
    
    # Create output dataframe - start with a copy of the phenotype dataframe
    residualized_df = pheno_df.copy()

    residualized_df = residualized_df[residualized_df[phenotype_cols[0]] >= -290]
    
    # For each phenotype, fit a regression model and calculate residuals
    for pheno in phenotype_cols:
        # Merge the dataframes for this analysis
        merged_df = pd.merge(
            pheno_df[[id_col, pheno]], 
            covar_df[[id_col] + covariate_cols],
            on=id_col,
            how='inner'
        )
        
        # Check if we have enough data after merging
        if len(merged_df) == 0:
            print(f"Warning: No matching IDs found for phenotype '{pheno}'. Skipping.")
            continue
        print(len(merged_df))
        
        # Remove rows with NaN in phenotype or any covariate
        valid_df = merged_df.dropna()
        
        if len(valid_df) == 0:
            print(f"Warning: No valid data for phenotype '{pheno}' after removing NaNs. Skipping.")
            continue
        
        # Get the data for the phenotype and covariates
        y = valid_df[pheno]
        X = valid_df[covariate_cols]
        
        # Add constant term for intercept
        X_with_const = sm.add_constant(X)
        
        # Fit the model
        model = sm.OLS(y, X_with_const).fit()
        
        # Calculate predicted values
        y_pred = model.predict(X_with_const)
        
        # Calculate residuals (observed - predicted)
        residuals = y - y_pred
        
        # Create temporary dataframe with IDs and residuals
        temp_df = pd.DataFrame({
            id_col: valid_df[id_col],
            pheno: residuals
        })
        
        # Update residualized dataframe with residuals
        # This keeps all original rows in pheno_df but updates only those with valid residuals
        residualized_df = residualized_df.set_index(id_col)
        temp_df = temp_df.set_index(id_col)
        residualized_df.update(temp_df)
        residualized_df = residualized_df.reset_index()
    
    return residualized_df

In [ ]:
covs_cols = ["type1_diabetes_true","type2_diabetes_true","MI_true","chronic_kidney_disease_true","beta_blocker_true","ace_inhibitor_true","arb_true","statin_true", "mra_true"]
phenos_cols = ["Mean_T1","T1_Standard_Deviation","T1_0.25th_Percentile","T1_1th_Percentile","T1_25th_Percentile","T1_50th_Percentile","T1_75th_Percentile","T1_99th_Percentile","T1_99.75th_Percentile"]

unet_t1_regressed_med = regress_out_covariates(unet_t1_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_all_phenotypes_PheWAS.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

phenos_cols = ["Latent_0","Latent_1","Latent_2","Latent_3","Latent_4","Latent_5","Latent_6","Latent_7","Latent_8","Latent_9","Latent_10","Latent_11","Latent_12","Latent_13","Latent_14","Latent_15"]

latent_dimensions_med = regress_out_covariates(latent_dimensions, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
latent_dimensions_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/latent_dimensions_16_allpatients_PheWAS.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

phenos_cols = ["mean","std","p0.25","p1","p25","p50","p75","p99","p99.5"]

unet_t1_2itr_regressed_med = regress_out_covariates(unet_t1_2itr_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_2itr_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_2itr_erroded.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

unet_t1_4itr_regressed_med = regress_out_covariates(unet_t1_4itr_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_4itr_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_4itr_erroded.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

unet_t1_6itr_regressed_med = regress_out_covariates(unet_t1_6itr_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_6itr_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_6itr_erroded.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

unet_t1_8itr_regressed_med = regress_out_covariates(unet_t1_8itr_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_8itr_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_8itr_erroded.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)

unet_t1_10itr_regressed_med = regress_out_covariates(unet_t1_10itr_regressed, medications_disease_covar, phenos_cols, covs_cols, id_col='IID')
unet_t1_10itr_regressed_med.to_csv("./data/shriya/SHMOLLI-output-unet-myocardium-update/T1_percentiles_PheWAS_10itr_erroded.no_outliers.medicalresiduals.qnorm.txt", sep='\t', index=False)